# Inspect AET Target

QA/QC of the combined actual-evapotranspiration calibration target
written by `targets/aet.py` to `<project>/targets/aet_targets.nc` (and
the optional `aet_targets_nn_filled.nc` companion).

The AET target is a per-HRU per-month **`(lower_bound, upper_bound)`
range in inches/day**, formed as the NaN-aware min / max across three
sources on a shared fabric:

- MOD16A2 v061 `ET_500m` (8-day kg/m²) → overlap-weighted monthly mm
  → inches/day (recipes §2).
- SSEBop `et` (mm/month, native) → inches/day.
- MWBM ClimGrid `aet` (mm/month, native) → inches/day.

The accompanying `n_sources` flag (0 / 1 / 2 / 3) records how many
sources contributed at each cell — the bound is NaN only when
`n_sources == 0`. The `nn_filled` companion file reuses the bounds but
additionally fills NaN HRUs from up to 10 nearest neighbours; the
`nn_filled` int8 flag (0 / 1) marks which HRU-times were filled.

On the gfv2 fabric the 21-year run (2000–2020) saw **zero all-NaN
cells**, so the NN-filled companion is functionally identical to the
raw target there. NN-fill remains useful for sparser fabrics or
shorter periods that don't have full source coverage.

This notebook checks:

- File schema and global metadata (period, fabric SHA, source string).
- Per-time `n_sources` coverage and where the gaps fall geographically.
- `lower_bound`, `upper_bound`, and range size at a representative time.
- CONUS area-weighted mean lower / upper time series.
- Representative-HRU time series across four AET regimes.
- NN-fill: which HRU-times were filled and whether the filled values
  preserve the lower / upper distribution.
- Order-of-magnitude sanity check against published CONUS AET.

Companion to:

- `inspect_consolidated_aet.ipynb` — pre-aggregation gridded NCs.
- `inspect_aggregated_aet.ipynb` — per-source HRU aggregates before
  multi-source combination (also where the canonical MOD16A2 overlap-
  weighted 8-day→monthly resample is documented).

## Conventions

- HRU dim name follows `fabric.id_col` from the project config
  (e.g. `nat_hru_id` for the GFv2 fabric, `nhm_id` elsewhere).
- Units are read from the **target NC variable attrs** (which are
  authoritative for the combined target — the catalog only documents
  per-source units, not the post-combination inches/day).
- `TARGET_TIME` (set below) drives the at-time choropleth panels.
  Defaults to July, the CONUS peak ET month, which gives the cleanest
  spread across sources (recipes §2: SSEBop/MWBM swing 6–11× across
  the seasonal cycle). Switch to January to inspect the cold-season
  low-ET regime instead.
- All maps are CONUS-Albers-equivalent only inside `area_weighted_mean`
  / `area_weighted_series` — the fabric is plotted in EPSG:4326 because
  the cost of reprojecting 360k polygons on every map is not worth it
  at inspection time.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    area_weighted_series,
    discover_target_nc,
    load_fabric,
    load_project_paths,
    load_representative_points,
    lookup_hrus_by_points,
    member_argextreme,
    member_categories,
    member_colors,
    member_frame_at_hru,
    member_frame_at_time,
    member_keys,
    n_sources_per_time,
    nan_hru_count,
    open_target_nc,
    plot_categorical_choropleth,
    plot_hru_choropleth,
    plot_member_panels,
    plot_nan_hrus,
    save_figure,
    select_month,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/targets/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

TARGET = "aet"
TARGET_YEAR = 2005
TARGET_MONTH = 7  # July -- peak CONUS ET; n_sources == 3 across interior
TARGET_TIME = f"{TARGET_YEAR}-{TARGET_MONTH:02d}-01"

# Four points spanning the wet/dry x cold/warm AET regimes.
REPRESENTATIVE_POINTS = load_representative_points(PROJECT_DIR, TARGET) or {
    "Iowa cropland (high summer ET, rainfed maize)": (-93.6, 42.0),
    "Southern Appalachians (Eastern broadleaf)": (-83.5, 35.5),
    "Phoenix metro (arid SW, low ET)": (-112.1, 33.4),
    "Olympic Peninsula (PNW conifer rainforest)": (-123.5, 47.8),
}

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)
id_dim = fabric_cfg["id_col"]

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
fabric_path = fabric_cfg['path']
print(f"Fabric: {fabric_path} ({len(fabric)} HRUs, id_col={id_dim!r})")
print(f"Target time: {TARGET_TIME}")

## Open and summarise the target NCs

`discover_target_nc` finds both the unfilled (`aet_targets.nc`) and
NN-filled (`aet_targets_nn_filled.nc`) variants if they exist.
Either may be `None` — this cell skips with a clear message and
downstream cells iterate only over what was loaded.


In [ ]:
raw_path, filled_path = discover_target_nc(project_dir, TARGET)

if raw_path is None:
    print(f"SKIP: {TARGET}_targets.nc not found at {project_dir / 'targets'}.")
    print("      Run `pixi run run-aet -- --project-dir <project>` first.")
    raise SystemExit

ds_raw = open_target_nc(raw_path)
print(f"Loaded raw target:    {raw_path.name}  ({raw_path.stat().st_size / 1e6:.1f} MB)")

ds_filled = None
if filled_path is not None:
    ds_filled = open_target_nc(filled_path)
    print(
        f"Loaded NN-filled:     {filled_path.name}  "
        f"({filled_path.stat().st_size / 1e6:.1f} MB)"
    )
else:
    print("NN-filled variant absent (set `nn_fill: true` in config to produce it).")


## Schema and global metadata

The global attrs carry the provenance you'd want at calibration time:
the period, the fabric SHA-256 (so a downstream consumer can detect a
fabric swap), the comma-separated `source` string, and the references
to TM 6-B10. Per-variable attrs carry the units (`inches/day` for the bounds,
`1` for the diagnostic flags).


In [ ]:
print(ds_raw)
print()
print("=== Global attrs ===")
for k, v in ds_raw.attrs.items():
    print(f"  {k:<20} {v}")

print()
print("=== Per-variable units ===")
for v in ("lower_bound", "upper_bound", "n_sources"):
    units = ds_raw[v].attrs.get("units", "(no units attr)")
    long_name = ds_raw[v].attrs.get("long_name", "")
    print(f"  {v:<14} units={units!r}  long_name={long_name!r}")


## Per-time coverage

`n_sources_per_time` returns the per-month count of HRUs at each flag
value (0 / 1 / 2 / 3 for a 3-source target). Two diagnostics:

- The `n=0` column is the count of all-NaN HRUs at that timestep —
  these are the cells the NN-fill targets. On the gfv2 21-year run
  this is zero everywhere.
- A jump in `n=2` (or drop in `n=3`) at a particular month flags a
  source whose period ended there. For AET, **MWBM ClimGrid ends
  2020-12**; SSEBop continues through 2023, MOD16A2 v061 through
  2025. Inside the 2000–2020 max-overlap window, all three
  contribute almost everywhere.

In [ ]:
cov = n_sources_per_time(ds_raw)
print(cov.describe().T[["mean", "std", "min", "max"]])

fig, ax = plt.subplots(figsize=(11, 4))
cov.plot(ax=ax)
ax.set_xlabel("Time")
ax.set_ylabel("HRU count")
ax.set_title(f"Per-month n_sources distribution — {TARGET} target")
ax.legend(title="flag value", loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_coverage_timeseries")
plt.show()


## n_sources map at TARGET_TIME

Categorical choropleth — colours match the `flag_values` attr on the
`n_sources` variable. NaN HRUs (where the time slice is missing) plot
in light grey; the legend reports the count of HRUs at each flag value
so coverage anomalies are visible at a glance.


In [ ]:
ns = select_month(ds_raw["n_sources"], TARGET_YEAR, TARGET_MONTH).to_pandas()

categories = {
    0: ("0 sources (all NaN)", "crimson"),
    1: ("1 source", "khaki"),
    2: ("2 sources", "lightgreen"),
    3: ("3 sources", "darkgreen"),
}

fig, ax = plt.subplots(figsize=(11, 7))
plot_categorical_choropleth(
    ax,
    fabric,
    ns,
    categories=categories,
    title=f"n_sources at {TARGET_TIME} — {TARGET} target",
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_n_sources_map")
plt.show()


## Lower / upper / range maps at TARGET_TIME

Three side-by-side panels:

- `lower_bound` — NaN-aware min across the three sources (inches/day).
- `upper_bound` — NaN-aware max across the three sources (inches/day).
- `range = upper - lower` — the per-HRU per-month spread the NHM
  calibrator is asked to land inside. Wide ranges = poor cross-source
  agreement; tight ranges = consensus among the LSMs at that cell.

Colour scale is anchored on the 2nd / 98th percentile of `upper_bound`
so the long tail of large-area HRUs (which dominate inches/day because inches/day is
volume-rate-per-HRU, not depth) doesn't flatten the rest of the map.


In [ ]:
lb = select_month(ds_raw["lower_bound"], TARGET_YEAR, TARGET_MONTH).to_pandas()
ub = select_month(ds_raw["upper_bound"], TARGET_YEAR, TARGET_MONTH).to_pandas()
rng = ub - lb

ub_finite = ub.dropna().values
vmin = float(np.percentile(ub_finite, 2))
vmax = float(np.percentile(ub_finite, 98))

units = ds_raw["lower_bound"].attrs.get("units", "inches/day")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
plot_hru_choropleth(
    axes[0], fabric, lb, vmin=vmin, vmax=vmax, cmap="YlGnBu",
    title=f"lower_bound\n{TARGET_TIME} | {units}", units=units,
)
plot_hru_choropleth(
    axes[1], fabric, ub, vmin=vmin, vmax=vmax, cmap="YlGnBu",
    title=f"upper_bound\n{TARGET_TIME} | {units}", units=units,
)
plot_hru_choropleth(
    axes[2], fabric, rng, vmin=0, vmax=vmax, cmap="OrRd",
    title=f"range = upper - lower\n{TARGET_TIME} | {units}", units=units,
)
fig.suptitle(f"{TARGET} target bounds — {TARGET_TIME}", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_bounds_map")
plt.show()

print(f"lower CONUS area-weighted mean: {area_weighted_mean(lb, fabric):,.2f} {units}")
print(f"upper CONUS area-weighted mean: {area_weighted_mean(ub, fabric):,.2f} {units}")
print(f"range CONUS area-weighted mean: {area_weighted_mean(rng, fabric):,.2f} {units}")


## NaN HRU coverage

HRUs where `n_sources == 0` (and therefore `lower_bound` / `upper_bound`
are NaN). These are the cells that nearest-neighbour fill targets — at
this time slice they show as red. A handful of NaN HRUs is normal for
the 2000–2010 window (typically very small fabric polygons whose source
grids didn't intersect at all); a large red blob means a source's
period or footprint isn't covering what we expect.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
plot_nan_hrus(
    ax, fabric, lb,
    title=(
        f"NaN HRUs (red) — {TARGET_TIME} | "
        f"{nan_hru_count(lb)} of {len(fabric)} "
        f"({100 * nan_hru_count(lb) / len(fabric):.2f}%)"
    ),
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_nan_map")
plt.show()


## CONUS area-weighted lower / upper time series

`area_weighted_series` reduces the (time, hru) bound arrays to a
per-month CONUS-mean, weighted by EPSG:5070 polygon area. Lower /
upper plotted together form an envelope — its width is the average
cross-source disagreement nationally.

The seasonal cycle is the dominant signal — peak in spring (snowmelt)
and a secondary autumn rise in the Pacific Northwest / Northeast.
Watch for:

- A flat lower bound near zero in summer (drought regimes).
- An expansion of the envelope in spring (snowmelt brings out
  ERA5-Land vs GLDAS vs MWBM physics differences).
- Step changes when a source's period boundary crosses the time axis.


In [ ]:
lb_series = area_weighted_series(ds_raw["lower_bound"], fabric, id_dim)
ub_series = area_weighted_series(ds_raw["upper_bound"], fabric, id_dim)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(
    lb_series.index, lb_series.values, ub_series.values,
    color="steelblue", alpha=0.25, label="lower–upper envelope",
)
ax.plot(lb_series.index, lb_series.values, color="steelblue", lw=1, label="lower")
ax.plot(ub_series.index, ub_series.values, color="darkblue", lw=1, label="upper")
ax.set_xlabel("Time")
ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
ax.set_title(f"{TARGET} target — CONUS area-weighted bounds")
ax.legend()
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_conus_series")
plt.show()


## Representative HRU time series

Four physiographically distinct points; `lookup_hrus_by_points` uses
`gpd.sjoin` to resolve each (lon, lat) to the containing HRU. The
lower / upper envelope at each point is the per-month spread the
calibrator targets locally — narrower envelopes = stronger inter-source
agreement at that cell.


In [ ]:
rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
print("Representative HRUs:", rep_hrus)

lb_at = ds_raw["lower_bound"].sel({id_dim: list(rep_hrus.values())})
ub_at = ds_raw["upper_bound"].sel({id_dim: list(rep_hrus.values())})

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
    lb_h = lb_at.sel({id_dim: hru_id}).values
    ub_h = ub_at.sel({id_dim: hru_id}).values
    t = pd.DatetimeIndex(lb_at.time.values)
    ax.fill_between(t, lb_h, ub_h, color="steelblue", alpha=0.25)
    ax.plot(t, lb_h, color="steelblue", lw=1, label="lower")
    ax.plot(t, ub_h, color="darkblue", lw=1, label="upper")
    ax.set_title(f"{label} (HRU {hru_id})")
    ax.set_ylabel(f"aet ({units})")
    ax.legend(fontsize=8)
fig.suptitle(f"{TARGET} target at representative HRUs", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_representative_series")
plt.show()


## Ensemble members at TARGET_TIME

One panel per contributing source, plus `ensemble_mean`, on a **single
shared color scale**. The shared scale is the point: per-panel
autoscaling renormalises every source to its own range, so sources that
disagree by a factor of three would render as near-identical maps. Panel
frames are tinted with each member's identity color, which is reused in
the driver map and the member series below.

Member names come from the file's `member_keys` attr, cross-checked
against the variables actually on disk — not from `config.yml`.

In [ ]:
keys = member_keys(ds_raw)

if not keys:
    print(f"SKIP: no ensemble members in this target, so no member maps.")
    print(f"      members_emitted={ds_raw.attrs.get('members_emitted', 'absent')!r}")
else:
    colors = member_colors(keys)
    units = ds_raw["lower_bound"].attrs.get("units", "1")
    frame = member_frame_at_time(ds_raw, keys, TARGET_TIME, id_dim)

    panels = {key: frame[key] for key in keys}
    panels["ensemble_mean"] = member_frame_at_time(
        ds_raw, ["ensemble_mean"], TARGET_TIME, id_dim
    )["ensemble_mean"]

    fig, (vmin, vmax) = plot_member_panels(
        fabric,
        panels,
        units=units,
        cmap="YlGnBu",
        colors=colors,
        suptitle=f"{TARGET} ensemble members at {TARGET_TIME} — shared scale",
    )
    save_figure(fig, f"{TARGET}_target_member_maps")
    plt.show()

    print(f"shared color scale: {vmin:,.4g} – {vmax:,.4g} {units}")
    print(f"{'member':<26}{'area-wtd mean':>16}{'NaN HRUs':>11}")
    for key in keys:
        col = frame[key]
        print(
            f"{key:<26}{area_weighted_mean(col, fabric):>16,.4g}"
            f"{int(col.isna().sum()):>11}"
        )

## Ensemble spread (`ensemble_std`) at TARGET_TIME

Where the sources disagree. `ensemble_std` is the population standard
deviation across the finite members, and it is **NaN wherever
`n_sources < 2`** — a deliberate mask, not a coverage gap: with one
finite source the population std is exactly `0`, and a calibration
weight built from it would read "only one source covered this cell" as
"perfect inter-source agreement." The legend names that class so the
grey is not left to interpretation.

In [ ]:
keys = member_keys(ds_raw)

if not keys:
    print(f"SKIP: no ensemble members in this target, so no ensemble_std map.")
    print(f"      members_emitted={ds_raw.attrs.get('members_emitted', 'absent')!r}")
else:
    units = ds_raw["lower_bound"].attrs.get("units", "1")
    std = member_frame_at_time(ds_raw, ["ensemble_std"], TARGET_TIME, id_dim)["ensemble_std"]

    fig, ax = plt.subplots(figsize=(11, 7))
    plot_hru_choropleth(
        ax,
        fabric,
        std,
        vmax=float(np.nanpercentile(std.values, 98)) if std.notna().any() else None,
        cmap="OrRd",
        title=f"ensemble_std at {TARGET_TIME} — {TARGET} target",
        units=units,
        nan_label="n_sources < 2 (std undefined)",
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_ensemble_std_map")
    plt.show()

    defined = int(std.notna().sum())
    print(f"ensemble_std defined at {defined}/{len(std)} HRUs "
          f"({100 * defined / len(std):.1f}%)")
    if defined:
        print(f"  median {float(np.nanmedian(std.values)):,.4g} {units}")
        print(f"  area-weighted mean {area_weighted_mean(std, fabric):,.4g} {units}")

## Which source drives each bound at TARGET_TIME

`upper_bound` is the NaN-aware max across members and `lower_bound` the
min, so at every HRU exactly one member *sets* each bound. These maps
name it.

Two classes are deliberately **not** members, and are drawn in grey:

- **no spread** — two or more sources are finite and all agree. A bare
  `argmax` would break that tie by silently returning the first member,
  which on a summer SWE day (every source reads 0 mm) would paint the
  entire map as one source's. Nothing drives a bound there.
- **single source** — only one member is finite, so there is no
  comparison to win. That cell's story is coverage; see the `n_sources`
  map above.

In [ ]:
keys = member_keys(ds_raw)

if not keys:
    print(f"SKIP: no ensemble members in this target, so no driver map.")
    print(f"      members_emitted={ds_raw.attrs.get('members_emitted', 'absent')!r}")
else:
    colors = member_colors(keys)
    frame = member_frame_at_time(ds_raw, keys, TARGET_TIME, id_dim)
    categories = member_categories(keys, colors)

    fig, axes = plt.subplots(1, 2, figsize=(22, 7))
    for ax, how, bound in ((axes[0], "max", "upper_bound"), (axes[1], "min", "lower_bound")):
        codes = member_argextreme(frame, how=how)
        plot_categorical_choropleth(
            ax,
            fabric,
            codes,
            categories=categories,
            title=f"source setting {bound} at {TARGET_TIME}",
        )
    fig.suptitle(f"{TARGET} — which source drives each bound", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_driver_map")
    plt.show()

    for how, bound in (("max", "upper_bound"), ("min", "lower_bound")):
        codes = member_argextreme(frame, how=how)
        share = codes.value_counts(dropna=False, normalize=True).sort_index()
        print(f"\n{bound} driver share:")
        for flag, pct in share.items():
            label = "no data" if pd.isna(flag) else categories[int(flag)][0]
            print(f"  {label:<30}{100 * pct:>6.1f}%")

## Member time series at representative HRUs

The same question as the driver map, asked at a point and through time:
each source's own trace inside the grey `lower`–`upper` envelope, with
`ensemble_mean` dashed. A bound that hugs one colored line across a
season is a bound set by that source.

In [ ]:
keys = member_keys(ds_raw)

if not keys:
    print(f"SKIP: no ensemble members in this target, so no member series.")
    print(f"      members_emitted={ds_raw.attrs.get('members_emitted', 'absent')!r}")
else:
    colors = member_colors(keys)
    units = ds_raw["lower_bound"].attrs.get("units", "1")
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)

    fig, axes = plt.subplots(2, 2, figsize=(15, 8.5), sharex=True)
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        members = member_frame_at_hru(ds_raw, keys, hru_id, id_dim)
        lo = ds_raw["lower_bound"].sel({id_dim: hru_id}).to_pandas()
        hi = ds_raw["upper_bound"].sel({id_dim: hru_id}).to_pandas()
        mean = ds_raw["ensemble_mean"].sel({id_dim: hru_id}).to_pandas()

        ax.fill_between(
            members.index, lo.values, hi.values,
            color="0.86", zorder=0, label="lower–upper envelope",
        )
        for key in keys:
            ax.plot(
                members.index, members[key].values,
                color=colors[key], lw=1.8, label=key, zorder=2,
            )
        ax.plot(
            mean.index, mean.values,
            color="0.25", lw=1.4, ls="--", label="ensemble_mean", zorder=3,
        )
        ax.set_title(f"{label} (HRU {hru_id})", fontsize=11)
        ax.set_ylabel(f"{TARGET} ({units})")

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="lower center", ncol=min(len(labels), 6), fontsize=9,
        frameon=False, bbox_to_anchor=(0.5, -0.04),
    )
    fig.suptitle(
        f"{TARGET} ensemble members at representative HRUs", fontsize=13, y=1.01
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_member_series")
    plt.show()

## NN-fill comparison

The companion `aet_targets_nn_filled.nc` reuses the same bounds but
fills NaN HRUs from up to `nn_max_candidates=10` nearest neighbours
(see `targets/run.py`). The `nn_filled` int8 flag marks each
HRU-time as `0` (not filled) or `1` (filled).

Two checks:

- **Filled-flag map at TARGET_TIME** — should match the NaN-HRU map
  one-for-one (any NaN that *can* be filled gets a `1`).
- **Distribution preservation** — the area-weighted CONUS mean of the
  filled bounds should track the unfilled mean closely (the NN fill
  is interpolating from immediate neighbours, not synthesising). A
  large jump indicates an over-aggressive fill.


In [ ]:
if ds_filled is None:
    print("SKIP: NN-filled variant not present.")
else:
    flag = select_month(ds_filled["nn_filled"], TARGET_YEAR, TARGET_MONTH).to_pandas()
    n_filled = int((flag == 1).sum())
    print(
        f"NN-filled at {TARGET_TIME}: {n_filled} HRUs filled "
        f"({100 * n_filled / len(fabric):.2f}%)"
    )

    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    plot_categorical_choropleth(
        axes[0], fabric, flag,
        categories={0: ("not filled", "lightgrey"), 1: ("filled", "crimson")},
        title=f"nn_filled flag at {TARGET_TIME}",
    )
    lb_f = select_month(ds_filled["lower_bound"], TARGET_YEAR, TARGET_MONTH).to_pandas()
    diff = (lb_f - lb).abs().fillna(lb_f)  # NaN in raw -> show filled value
    plot_hru_choropleth(
        axes[1], fabric, diff, vmin=0, vmax=vmax / 4, cmap="OrRd",
        title=f"|lower_bound_filled - lower_bound_raw| at {TARGET_TIME}",
        units=units,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_map")
    plt.show()

    lb_f_series = area_weighted_series(ds_filled["lower_bound"], fabric, id_dim)
    ub_f_series = area_weighted_series(ds_filled["upper_bound"], fabric, id_dim)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(lb_series.index, lb_series.values, color="steelblue", lw=1, label="lower (raw)")
    ax.plot(lb_f_series.index, lb_f_series.values, color="steelblue", lw=1, ls="--", label="lower (filled)")
    ax.plot(ub_series.index, ub_series.values, color="darkblue", lw=1, label="upper (raw)")
    ax.plot(ub_f_series.index, ub_f_series.values, color="darkblue", lw=1, ls="--", label="upper (filled)")
    ax.set_xlabel("Time")
    ax.set_ylabel(f"Area-weighted CONUS mean ({units})")
    ax.set_title(f"{TARGET} target — raw vs NN-filled CONUS series")
    ax.legend()
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_series")
    plt.show()


## Order-of-magnitude sanity check

Compute the CONUS area-weighted mean of `lower_bound` and
`upper_bound` at `TARGET_TIME` and compare to a published reference.
TM 6-B10 cites CONUS-mean annual AET around **600–700 mm/yr**
(≈ 23.6–27.6 in/yr, ≈ 0.065–0.076 in/day on an annualised basis). At
the peak July month, the daily rate of the **upper** bound on CONUS
should comfortably exceed the annual mean, while the **lower** bound
may sit near or below it depending on which source is dragging
(MOD16A2 v061 typically sets the lower bound in summer per recipes
§2).

A summed bound that is *orders of magnitude* off the reference is a
smoking gun for a missed conversion factor (per
`feedback_validate_magnitudes.md`).

In [ ]:
# CONUS area-weighted means at TARGET_TIME; compare to long-term
# annual-mean references (in/day on the annualised basis).
lb_mean = area_weighted_mean(lb, fabric)
ub_mean = area_weighted_mean(ub, fabric)

# References (TM 6-B10 / standard CONUS AET climatology):
REF_ANNUAL_MM_LO, REF_ANNUAL_MM_HI = 500.0, 750.0  # mm/yr (broad)
ref_lo_in_day = (REF_ANNUAL_MM_LO / 25.4) / 365.25
ref_hi_in_day = (REF_ANNUAL_MM_HI / 25.4) / 365.25

print(f"CONUS area-weighted lower_bound at {TARGET_TIME}: {lb_mean:.4f} in/day")
print(f"CONUS area-weighted upper_bound at {TARGET_TIME}: {ub_mean:.4f} in/day")
print(f"Reference (annualised, 500–750 mm/yr): {ref_lo_in_day:.4f} – {ref_hi_in_day:.4f} in/day")
ratio_upper = 2 * ub_mean / (ref_lo_in_day + ref_hi_in_day)
print(f"Ratio (upper / annual-mean midpoint): {ratio_upper:.2f}x (expect 1.0–2.5x at July peak)")

In [ ]:
ds_raw.close()
if ds_filled is not None:
    ds_filled.close()
